# Error Rate Analysis – Vanguard A/B Test
Builds the master dataset (`df_cd`) with per-client error flags, error counts,
completion times, and group labels.  Exports two CSVs consumed by Tableau and
by `stats_claire.ipynb`.

**Outputs**
- `../data/clean/df_alldata.csv` – full enriched client table
- `../data/clean/funnel_by_step.csv` – step-level reach counts & rates
- `../data/clean/errors_by_transition.csv` – transition-level error counts & rates


## 1  Imports

In [ ]:
import pandas as pd
import numpy as np
import yaml

## 2  Load config

In [ ]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

config

## 3  Load data
`df_w` – full web-log events; `df_test` / `df_cont` – group rosters; `df_cd` – client demographics.

In [ ]:
df_w    = pd.read_csv(config['data']['clean']['file6'], quotechar='"')
df_test = pd.read_csv(config['data']['clean']['file3'], quotechar='"')
df_cont = pd.read_csv(config['data']['clean']['file4'], quotechar='"')
df_cd   = pd.read_csv(config['data']['clean']['file1'], quotechar='"')

## 4  Split event log by group

In [ ]:
df_wtest    = df_w[df_w.client_id.isin(df_test.client_id)]
df_wcontrol = df_w[df_w.client_id.isin(df_cont.client_id)]

## 5  Helper functions

In [ ]:
def contains_step_error(lst, sub):
    """
    Slide a window of len(sub) across `lst` and return True if `sub` appears
    as a contiguous subsequence.  Used to detect backwards-navigation errors
    (e.g. step_1 → step_2 → step_1 means the user regressed from step_2).
    """
    n = len(sub)
    return any(lst[i:i + n] == sub for i in range(len(lst) - n + 1))

In [ ]:
STEPS = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

def get_completion_rates(df):
    """
    Return a dict {step: unique_client_count} for each process step.
    Denominator is intentionally left out here; normalisation is done
    downstream so the raw counts stay reusable.
    """
    total = df['client_id'].nunique()
    return {step: df[df['process_step'] == step]['client_id'].nunique()
            for step in STEPS}

In [ ]:
# Regression patterns: A → B → A means the user bounced back from B to A.
ERROR_PATTERNS = {
    's_1': ['start',  'step_1',  'start'],
    '1_2': ['step_1', 'step_2',  'step_1'],
    '2_3': ['step_2', 'step_3',  'step_2'],
    '3_c': ['step_3', 'confirm', 'step_3'],
}

def get_error_rate(df):
    """
    Scan each client's ordered step sequence for any regression pattern.

    Returns
    -------
    error_counts : dict  {pattern_key: count_of_clients_who_hit_it,
                          'err_counts': total_clients_with_any_error}
    results      : dict  {pattern_key: set_of_client_ids,
                          'err_counts': set_of_all_error_client_ids}

    Notes
    -----
    Raw counts are returned (not rates) because the appropriate denominator
    differs by transition – e.g. only clients who reached step_3 can
    produce a 3→confirm regression.
    """
    results   = {key: set() for key in ERROR_PATTERNS}
    err_ids   = set()

    for client_id, group in df.groupby('client_id'):
        step_list = group['process_step'].tolist()
        for key, pattern in ERROR_PATTERNS.items():
            if contains_step_error(step_list, pattern):
                results[key].add(client_id)
                err_ids.add(client_id)

    error_counts = {key: len(ids) for key, ids in results.items()}
    error_counts['err_counts'] = len(err_ids)
    results['err_counts']      = err_ids
    return error_counts, results

## 6  Compute funnel counts and error counts

In [ ]:
finished_cont = get_completion_rates(df_wcontrol)
finished_test = get_completion_rates(df_wtest)

error_cont, res_cont = get_error_rate(df_wcontrol)
error_test, res_test = get_error_rate(df_wtest)

## 7  Build enriched client table (`df_cd`)

In [ ]:
# Tag each client with their experimental group
df_cd_c = df_cd[df_cd.client_id.isin(df_cont.client_id)].copy()
df_cd_t = df_cd[df_cd.client_id.isin(df_test.client_id)].copy()

df_cd_c['group'] = 'control'
df_cd_t['group'] = 'test'

In [ ]:
# Parse datetimes (needed for completion-time calculation)
df_wcontrol['date_time'] = pd.to_datetime(df_wcontrol['date_time'])
df_wtest['date_time']    = pd.to_datetime(df_wtest['date_time'])

In [ ]:
def get_completion_time(group):
    """
    Per-client completion time = latest confirm timestamp − latest start timestamp.
    Returns None if the client never started or never confirmed.
    """
    starts   = group[group['process_step'] == 'start']['date_time']
    confirms = group[group['process_step'] == 'confirm']['date_time']
    if starts.empty or confirms.empty:
        return None
    return confirms.max() - starts.max()


def add_error_cols(df_cd, df_w, results):
    """
    Merge completion time and error flags into the client demographics table.

    New columns
    -----------
    completion_time : timedelta  (NaT if client never completed)
    made_error      : bool       True if client hit any regression pattern
    n_errors        : int        count of distinct patterns triggered
    """
    PATTERN_KEYS = ['s_1', '1_2', '2_3', '3_c']
    error_ids    = results['err_counts']

    completion_times = (
        df_w.groupby('client_id')
            .apply(get_completion_time)
            .reset_index()
            .rename(columns={0: 'completion_time'})
    )

    df_cd = df_cd.merge(completion_times, on='client_id', how='left')
    df_cd['made_error'] = df_cd['client_id'].isin(error_ids)
    df_cd['n_errors']   = df_cd['client_id'].apply(
        lambda x: sum(x in results[k] for k in PATTERN_KEYS)
    )
    return df_cd

In [ ]:
df_cd_c = add_error_cols(df_cd_c, df_wcontrol, res_cont)
df_cd_t = add_error_cols(df_cd_t, df_wtest,    res_test)

# Combine and restrict to clients present in the event log
df_cd = pd.concat([df_cd_c, df_cd_t], ignore_index=True)
df_cd = df_cd[df_cd['client_id'].isin(df_w['client_id'])]

In [ ]:
# Derived time columns
df_cd['completion_seconds']  = df_cd['completion_time'].dt.total_seconds()
df_cd['completion_hrs_mins'] = (
    pd.Timestamp('1900-01-01') + df_cd['completion_time']
).dt.time

# Boolean completion flag (used in stats notebook)
df_cd['completed'] = df_cd['completion_time'].notna()

df_cd.head()

## 8  Build and export Tableau CSVs

In [ ]:
# ── Funnel CSV ────────────────────────────────────────────────────────────────
ctrl_counts = [finished_cont[s] for s in STEPS]
test_counts = [finished_test[s] for s in STEPS]

funnel_rows = []
for step, c, t in zip(STEPS, ctrl_counts, test_counts):
    funnel_rows.append({'step': step, 'group': 'control',
                        'clients_reached': c,
                        'completion_rate': round(c / ctrl_counts[0], 4)})
    funnel_rows.append({'step': step, 'group': 'test',
                        'clients_reached': t,
                        'completion_rate': round(t / test_counts[0], 4)})

df_funnel = pd.DataFrame(funnel_rows)
# df_funnel.to_csv('../data/clean/funnel_by_step.csv', index=False, encoding='utf-8')

df_funnel

In [ ]:
# ── Errors CSV ───────────────────────────────────────────────────────────────
TRANSITIONS   = ['start→step_1', 'step_1→step_2', 'step_2→step_3', 'step_3→confirm']
err_ctrl_list = [error_cont[k] for k in ['s_1', '1_2', '2_3', '3_c']]
err_test_list = [error_test[k] for k in ['s_1', '1_2', '2_3', '3_c']]

error_rows = []
for trans, c, t in zip(TRANSITIONS, err_ctrl_list, err_test_list):
    error_rows.append({'transition': trans, 'group': 'control',
                       'error_count': c,
                       'error_rate':  round(c / ctrl_counts[0], 4)})
    error_rows.append({'transition': trans, 'group': 'test',
                       'error_count': t,
                       'error_rate':  round(t / test_counts[0], 4)})

df_errors = pd.DataFrame(error_rows)
# df_errors.to_csv('../data/clean/errors_by_transition.csv', index=False, encoding='utf-8')

df_errors

In [ ]:
# ── Full enriched client table ────────────────────────────────────────────────
# df_cd.to_csv('../data/clean/df_alldata.csv', index=False, encoding='utf-8')